In [1]:
!pip install ddgs trafilatura -q
!pip install openai-agents

  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ---------------------------------------- 0.0/816.1 kB ? eta -:--:--
   ---------------------------------------- 816.1/816.1 kB 11.8 MB/s  0:00:00
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 3.8/3.8 MB 35.9 MB/s  0:00:00
Using cached cffi-2.0.0-cp314-cp314-win_amd64.whl (185 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 9.7/9.7 MB 65.8 MB/s  0:00:00
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

   ----------------------------------------  0/11 [pywin32]
   ----------------------------------------  0/11 [pywin32]
   ----------------------------------------  0/11 [pywin32]
   ---------------

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from pprint import pprint
from IPython.display import Markdown, display

from agents import Agent, Runner, function_tool



load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing!")
else:
    print("Key is: " + OPENAI_API_KEY[:8])

client = OpenAI(api_key=OPENAI_API_KEY)

Key is: sk-proj-


In [ ]:
@function_tool
def search_web(query):
    ddgs = DDGS()

    results = ddgs.text(query, max_result = 5)
    #pprint(f"Got results: {results}")
    return json.dumps(results, indent=2)

In [27]:
res = search_web("Ai in 2030")
print(res)

[
  {
    "title": "AI in 2025",
    "href": "https://en.wikipedia.org/wiki/AI_in_2025",
    "body": "The following is a list of events of the year 2025 in artificial intelligence."
  },
  {
    "title": "What will AI look like in 2030? | Epoch AI",
    "href": "https://epoch.ai/blog/what-will-ai-look-like-in-2030",
    "body": "What will happen if AI scaling persists to 2030? We are releasing a report that examines what this scale-up would involve in terms of compute, investment, data, hardware, and energy."
  },
  {
    "title": "How will Artificial Intelligence Affect Jobs 2026-2030 | Nexford University",
    "href": "https://www.nexford.edu/insights/how-will-ai-affect-jobs",
    "body": "AI will be taking some jobs, but it will be creating new ones! Here are the most likely jobs that artificial intelligence will affect from 2026-2030"
  },
  {
    "title": "[2211.06318] Artificial Intelligence and Life in 2030: The One Hundred...",
    "href": "https://arxiv.org/abs/2211.06318",
  

In [14]:
@function_tool
def fetch_url(url: str):

    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(text)
        if text: 
            print(f" \u2705 Got Text")
            return text
    print(f"\u274C fail to fetch text from URL {url}. ")
    return (f"Could not extract text from url {url}. Try different source.")


## Step 4: System Prompt

In [28]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief
 
Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

In [29]:
JUDGE_MODEL = "gpt-4.1"
MODEL = "gpt-4.1-nano"

research_agent = Agent(
    name="ResearchAgent",
    instructions=RESEARCH_AGENT_PROMPT,
    model = MODEL,
    tools = [search_web, fetch_url]
)

In [30]:
result = await Runner.run(
    research_agent,
    input="Research the following topic and produce a comprehensive brief on Ai in Healthcare in 2030"
)

Error getting response: Unknown tool type: <class 'function'>, tool. (request_id: None)


UserError: Unknown tool type: <class 'function'>, tool